# BWE Stufe 1 — Regressions-Training (Kaggle/GPU)

**Auf Kaggle ausführen.** Im rechten Settings-Panel des Notebooks:

1. **Accelerator → GPU** (z. B. T4 ×2 oder P100). Ist die Auswahl ausgegraut oder bleibt
   die GPU-Prüfung unten bei `GPUs: []`: meist fehlt die **Telefon-Verifizierung**
   (Profilbild → *Settings* → *Phone verification*) — sie schaltet GPU **und** Internet
   frei. Sonst: Wochen-Kontingent (~30 GPU-h) erschöpft, oder laufende Session erst
   stoppen, dann Accelerator umstellen.
2. **Internet → On** (Schalter im rechten Panel, zwischen *Environment* und *Tags*;
   braucht ebenfalls Telefon-Verifizierung). Ohne Internet scheitert `git clone`
   („Could not resolve host: github.com").
3. **Add Data → MUSDB18-HQ** einbinden (z. B. `quanglvitlm/musdb18-hq`) und in Zelle 2
   `BWE_DATA_ROOT` auf den Ordner zeigen lassen, der `train/` und `test/` enthält
   (exakten Pfad im *Input*-Panel ablesen; ein evtl. `val/`-Ordner wird ignoriert).

Repo ist public → Clone ohne Token. Lokal nicht ausführbar (klont Repo, braucht GPU + Datensatz).

In [ ]:
# GPU prüfen
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Code holen & installieren

TF ist auf Kaggle vorinstalliert → `--no-deps`, damit die GPU-Build nicht ersetzt wird; nur die Audio-Pakete nachinstallieren.

In [ ]:
REPO = "https://github.com/DanyelRychter/bwe_dnn.git"
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo

import subprocess
subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/bwe_dnn"], check=True)
%pip install -q -e /kaggle/working/bwe_dnn --no-deps
%pip install -q librosa soundfile soxr

## 2. Pfade setzen (VOR dem bwe-Import!)

`BWE_DATA_ROOT` = Ordner, der `train/` und `test/` enthält. Ein evtl. vorhandener `val/`-Ordner wird ignoriert (kanonischer Split via Track-Namen).

In [ ]:
import os
# >>> An die tatsächliche Dataset-Struktur anpassen: <<<
os.environ["BWE_DATA_ROOT"] = "/kaggle/input/datasets/quanglvitlm/musdb18-hq"     # muss train/ und test/ enthalten
os.environ["BWE_CKPT_ROOT"] = "/kaggle/working/bwe_runs"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import sys; sys.path.insert(0, "/kaggle/working/bwe_dnn")
from bwe import config as cfg
print(cfg.summary())

In [ ]:
# Datensatz-Check: erwartete 86/14/50 (bzw. tatsächliche Zahlen)
from bwe.data import splits as SP
for s in SP.SPLIT_NAMES:
    print(f"{s:6s}: {len(SP.get_split(s)):3d} Tracks")

## 2b. 32-kHz-Cache (empfohlen fürs Volltraining)

Im Subset-Lauf war die **GPU nur ~17 % ausgelastet**: die `tf.data`-Pipeline resampelt jeden
Crop on-the-fly von 44,1 kHz auf 32 kHz auf der CPU → das Training ist I/O-/CPU-Bound. Die
Funktion `use_resample_cache` (aus `bwe/data/cache.py`) resampelt die 4 Stems **aller**
`train/`- und `test/`-Tracks **einmal** nach `/kaggle/working/musdb18hq_32k` und biegt
`cfg.DATA_ROOT` darauf. Danach gilt im Training `sr == cfg.SR`, der Resample-Zweig entfällt
→ **GPU-Bound**.

- **Stereo bleibt erhalten** → Kanal-Augmentation (`mean`/`left`/`right`) unverändert.
- **`mixture.wav` wird nicht gecacht** (Mixture = Summe der Stems) → spart Platz und Zeit.
- **Resumebar:** bereits gecachte Dateien werden übersprungen.
- **Logik ausgelagert** nach `bwe/data/cache.py` → in allen Trainings-Notebooks (GAN etc.)
  per `from bwe.data.cache import use_resample_cache` nutzbar, ohne den Code zu kopieren.
- **Commit-Modus:** `/kaggle/working` ist je Lauf frisch → der Cache wird jedes Mal neu
  gebaut (~13 min für 150 Tracks bei P100). Für **Subset-Läufe im Edit-Modus** diese Zelle
  einfach **überspringen** (20 Tracks on-the-fly sind schnell genug).

In [ ]:
# 32-kHz-Cache: alle Stems einmal resampeln und cfg.DATA_ROOT darauf umbiegen.
# Logik ausgelagert nach bwe/data/cache.py -> in allen Trainings-Notebooks wiederverwendbar.
# Danach gilt im Training sr==cfg.SR -> kein On-the-fly-Resampling -> GPU-Bound.
from bwe.data.cache import use_resample_cache
from bwe.data import splits as SP

use_resample_cache(os.environ["BWE_DATA_ROOT"], "/kaggle/working/musdb18hq_32k")
for s in SP.SPLIT_NAMES:                        # Gegencheck aus dem Cache: erwartet 86/14/50
    print(f"  {s:6s}: {len(SP.get_split(s)):3d} Tracks")

## 3. Subset-Training (Architektur/Hyperparameter)

Wenige Tracks → schnell prüfen, dass Val-LSD-HF unter die Copy-Up-Baseline sinkt.

In [ ]:
from bwe.train.regression import train
model, hist = train(run="reg_subset", mode="subset", batch_size=16, epochs=30, limit=20)

## 4. Volltraining (Cold-Start oder Resume)

Eine Zelle, drei Fälle — funktioniert im **Commit-Modus** out-of-the-box (das ganze
Notebook läuft top-to-bottom, kein einzelnes Zell-Ausführen nötig). Gesteuert über
**eine** Variable `CKPT_SRC`:

- **`CKPT_SRC = None`** → **Cold-Start** von Null. Standard für den ersten Volllauf.
- **`CKPT_SRC` zeigt auf ein Dataset mit `backup/`** → **exaktes Resume** (Optimizer +
  Epochenzähler via `BackupAndRestore`).
- **`CKPT_SRC` nur mit `best.weights.h5`** → **Warm-Start** (gelernte Gewichte, Adam und
  Epochenzähler starten neu).

> Workflow: **Subset** im *Edit-Modus* (Zelle 3), **Volllauf** im *Commit-Modus*. Vor dem
> Commit die Subset-Zelle auskommentieren/entfernen, dann *Save & Run All (Commit)*.

### Resume im Commit-Modus (exakt, inkl. `backup/`)
Kaggle sichert bei einem Commit `/kaggle/working` automatisch als **Output der Version** —
inklusive `bwe_runs/reg_full/backup/`. Damit klappt exaktes Resume **ohne** einzelne Zellen
neu auszuführen:
1. **Add Input → Your Work** → den Output der vorigen Commit-Version anhängen (oder den
   Ordner als Dataset hochladen).
2. `CKPT_SRC` auf den Pfad im *Input*-Panel setzen (`/kaggle/input/<slug>`, ggf. mit dem
   Unterordner `bwe_runs/reg_full`).
3. Erneut committen → liegt `backup/` darin, läuft es **exakt an der letzten Epoche**
   weiter; sonst Warm-Start ab `best.weights.h5`.

> Damit ein Resume **über die ursprünglichen Epochen hinaus** weiterläuft, ggf. `epochs`
> erhöhen (sonst ist nach `cfg.EPOCHS` Schluss). Der **EarlyStopping-Status** steht jetzt
> explizit im Log (`>>> EarlyStopping: ... <<<` bzw. `>>> Training regulär beendet ... <<<`),
> sodass ein sauberer Stopp von einem echten Abbruch unterscheidbar ist.

In [ ]:
# Volltraining -- Cold-Start ODER Resume, je nach CKPT_SRC (siehe Markdown oben).
#   CKPT_SRC = None                       -> Cold-Start (von Null)
#   CKPT_SRC mit .../backup/              -> exaktes Resume (Optimizer + Epoche)
#   CKPT_SRC nur mit best.weights.h5      -> Warm-Start (gelernte Gewichte)
from pathlib import Path
from bwe.train.regression import train_resumable

# >>> Zum Fortsetzen: angehaengtes Checkpoint-Dataset eintragen, sonst None lassen. <<<
CKPT_SRC = None   # z.B. Path("/kaggle/input/bwe-reg-full-ckpt")

model, hist = train_resumable(run="reg_full", mode="full", ckpt_src=CKPT_SRC,
                              batch_size=cfg.BATCH_SIZE, epochs=cfg.EPOCHS)

## 5. Lernkurve + Persistenz

Gewichte unter `/kaggle/working/bwe_runs/reg_full/` (best.weights.h5 + generator.weights.h5).
Für Phase D / das Ergebnis-Notebook herunterladen oder als Kaggle-Output-Dataset sichern.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv("/kaggle/working/bwe_runs/reg_full/log.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(log["epoch"], log["loss"], label="train"); ax[0].plot(log["epoch"], log["val_loss"], label="val")
ax[0].set_title("RI+Mag-Loss"); ax[0].set_xlabel("Epoche"); ax[0].legend()
ax[1].plot(log["epoch"], log["lsd_hf"], label="train"); ax[1].plot(log["epoch"], log["val_lsd_hf"], label="val")
ax[1].set_title("LSD-HF [dB]"); ax[1].set_xlabel("Epoche"); ax[1].legend()
plt.tight_layout(); plt.show()